# S4 · AndinaLog 03B · Notebook 2 · Tratamiento de pedidos WMS

Este notebook consume las salidas del notebook 1 y `catalogo_reglas_tratamiento.csv`. Solo ejecuta reglas cuyo estado sea `APROBADA`. Conserva los catorce campos Bronze, registra cada decisión y exporta `andinalog_wms_orders_silver.csv` con solo las filas utilizables y las columnas de negocio preparadas. También nunca obliga a una fila a salir de cuarentena.

Con el catálogo entregado, la única regla aprobada es la exclusión de copias exactamente iguales de un pedido duplicado. Las normalizaciones de identificadores, fechas, cantidades y tiempos permanecen pendientes hasta que sean acordadas.


## 1 · Configuración y rutas

En local puede ejecutarse desde cualquier carpeta dentro del repositorio. En Colab, ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `S4/`. Cada ejecución reemplaza las salidas anteriores.


In [2]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd

ENTORNO = "auto"  # "auto", "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
VERSION_TRATAMIENTO = "GIAD-M3-S4-WMS-orders-tratamiento-v1"

COLUMNAS_ORIGINALES = [
    "order_id", "cliente_id", "producto_id", "fecha_despacho", "centro_distribucion",
    "camion_id", "chofer_id", "cantidad_solicitada", "cantidad_entregada",
    "tiempo_entrega_prometido_hrs", "tiempo_entrega_real_hrs",
    "otif_on_time", "otif_in_full", "otif",
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS).is_dir() and (carpeta / "S4").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

def configurar_rutas():
    entorno = ENTORNO
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser auto, local o drive")
    n1 = raiz / "S4" / "andinalog_wms_orders" / "notebook1" / "salidas"
    n2 = raiz / "S4" / "andinalog_wms_orders" / "notebook2"
    rutas = {
        "bronze": raiz / "datasets" / CARPETA_DATASETS / "andinalog_wms_orders.csv",
        "diagnosticado": n1 / "andinalog_wms_orders_diagnosticado.csv",
        "problemas": n1 / "andinalog_wms_orders_problemas.csv",
        "reporte_n1": n1 / "andinalog_wms_orders_reporte_calidad.csv",
        "catalogo": n2 / "catalogo_reglas_tratamiento.csv",
        "salidas": n2 / "salidas",
    }
    faltantes = [str(p) for k,p in rutas.items() if k != "salidas" and not p.is_file()]
    if faltantes:
        raise FileNotFoundError("Faltan archivos requeridos:\n" + "\n".join(faltantes))
    return rutas

RUTAS = configurar_rutas()
RUTAS


{'bronze': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/datasets/AndinaLog_03B_Bronce/andinalog_wms_orders.csv'),
 'diagnosticado': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_wms_orders/notebook1/salidas/andinalog_wms_orders_diagnosticado.csv'),
 'problemas': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_wms_orders/notebook1/salidas/andinalog_wms_orders_problemas.csv'),
 'reporte_n1': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_wms_orders/notebook1/salidas/andinalog_wms_orders_reporte_calidad.csv'),
 'catalogo': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_wms_orders/notebook2/catalogo_reglas_tratamiento.csv'),
 'salidas': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_wms_orders/notebook2/salidas')}

## 2 · Carga, trazabilidad y catálogo

Se comprueba que el archivo diagnosticado corresponde al mismo Bronze mediante la huella SHA-256 registrada por el notebook 1.


In [3]:
def cargar_csv(ruta):
    return pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)

bronze = cargar_csv(RUTAS["bronze"])
diagnosticado = cargar_csv(RUTAS["diagnosticado"])
problemas = cargar_csv(RUTAS["problemas"])
reporte_n1 = cargar_csv(RUTAS["reporte_n1"])
catalogo = cargar_csv(RUTAS["catalogo"])

hash_bronze = hashlib.sha256(RUTAS["bronze"].read_bytes()).hexdigest()
hash_reportado = reporte_n1.loc[reporte_n1["metrica"].eq("sha256_bronze"), "valor"].iloc[0]
assert hash_bronze == hash_reportado, "El Bronze no corresponde al diagnóstico del notebook 1"
assert list(bronze.columns) == COLUMNAS_ORIGINALES
assert diagnosticado[COLUMNAS_ORIGINALES].equals(bronze[COLUMNAS_ORIGINALES])
assert catalogo["regla_id"].is_unique
assert catalogo["estado"].isin(["APROBADA", "PENDIENTE"]).all()

reglas_aprobadas = set(catalogo.loc[catalogo["estado"].eq("APROBADA"), "regla_id"])
print("Reglas aprobadas:", sorted(reglas_aprobadas))
display(catalogo)


Reglas aprobadas: ['DUPLICADO_IDENTICO']


,regla_id,columna_afectada,codigo_error_o_unidad,tratamiento_propuesto,estado,evidencia_acuerdo,validacion_requerida
0,DUPLICADO_IDENTICO,order_id,DUPLICADO,Conservar la primera fila como canónica y excl...,APROBADA,Criterio conservador de duplicados acordado pr...,Mismo order_id y las 14 columnas originales id...
1,NORMALIZAR_ORDER_ID,order_id,FORMATO_INVALIDO,Quitar espacios externos y convertir a mayúsculas,PENDIENTE,,El resultado debe cumplir ORD-2026-##### y no ...
2,NORMALIZAR_CLIENTE_ID,cliente_id,FORMATO_INVALIDO,Quitar espacios externos y convertir a mayúsculas,PENDIENTE,,El resultado debe cumplir CLI-### y no alterar...
3,NORMALIZAR_PRODUCTO_ID,producto_id,FORMATO_INVALIDO,Quitar espacios externos y convertir a mayúsculas,PENDIENTE,,El resultado debe cumplir PROD-### y no altera...
4,FECHA_DMY_A_ISO,fecha_despacho,FECHA_INVALIDA,Convertir DD/MM/AAAA HH:MM a AAAA-MM-DD HH:MM:SS,PENDIENTE,,La fecha debe ser válida y la conversión inequ...
5,FECHA_CALENDARIO_INVALIDA,fecha_despacho,FECHA_INVALIDA,Mantener en cuarentena sin inventar una fecha,PENDIENTE,,Fecha real recuperada de una fuente autorizada
6,CANTIDAD_TEXTO_A_NUMERO,cantidad_solicitada,NO_NUMERICA,Convertir palabras mediante un diccionario apr...,PENDIENTE,,Correspondencia textual aprobada y cantidad po...
7,CANTIDAD_ENTREGADA_FALTANTE,cantidad_entregada,FALTANTE,Mantener en cuarentena sin imputación automática,PENDIENTE,,Cantidad real recuperada de una fuente autorizada
8,TIEMPO_REAL_NEGATIVO,tiempo_entrega_real_hrs,NEGATIVO,Mantener en cuarentena sin cambiar signo ni im...,PENDIENTE,,Tiempo real recuperado o regla de corrección a...
9,DUPLICADO_FECHA_EQUIVALENTE,order_id,DUPLICADO,Comparar fechas equivalentes y elegir una fila...,PENDIENTE,,Las 14 columnas deben ser equivalentes después...


## 3 · Preparación y decisiones sobre duplicados

Las columnas terminadas en `_preparado` son el área de trabajo. Los campos Bronze permanecen intactos. Una copia exacta se conserva en el archivo tratado para auditoría, pero queda marcada como no utilizable.


In [4]:
def preparar_base(df):
    salida = df.copy(deep=True)
    for columna in COLUMNAS_ORIGINALES:
        salida[f"{columna}_preparado"] = salida[columna]
    salida["registro_canonico"] = True
    salida["excluido_como_copia"] = False
    return salida

def decidir_duplicados_exactos(df):
    decisiones = []
    original = df[COLUMNAS_ORIGINALES]
    for order_id, indices in df.groupby(df["order_id"].str.strip(), sort=False).groups.items():
        indices = list(indices)
        if len(indices) < 2:
            continue
        canonica = indices[0]
        for idx in indices[1:]:
            exacta = original.loc[idx].equals(original.loc[canonica])
            if exacta and "DUPLICADO_IDENTICO" in reglas_aprobadas:
                decision = "COPIA_EXCLUIDA"
                regla = "DUPLICADO_IDENTICO"
                justificacion = "Las 14 columnas originales coinciden con la primera fila del pedido"
            else:
                decision = "PENDIENTE"
                regla = "DUPLICADO_CONFLICTIVO"
                justificacion = "Las filas no son idénticas o la regla aplicable no está aprobada"
            decisiones.append({
                "fila_bronze": int(df.at[idx, "fila_bronze"]),
                "order_id": df.at[idx, "order_id"],
                "fila_canonica": int(df.at[canonica, "fila_bronze"]),
                "decision": decision, "regla_id": regla, "justificacion": justificacion,
            })
    return pd.DataFrame(decisiones, columns=["fila_bronze","order_id","fila_canonica","decision","regla_id","justificacion"])

df_trabajo = preparar_base(diagnosticado)
decisiones_duplicados = decidir_duplicados_exactos(df_trabajo)
filas_excluidas = set(decisiones_duplicados.loc[decisiones_duplicados["decision"].eq("COPIA_EXCLUIDA"), "fila_bronze"].astype(int))
df_trabajo["excluido_como_copia"] = df_trabajo["fila_bronze"].astype(int).isin(filas_excluidas)
df_trabajo.loc[df_trabajo["excluido_como_copia"], "registro_canonico"] = False
print("Copias exactas excluidas:", len(filas_excluidas))
display(decisiones_duplicados.head())


Copias exactas excluidas: 49


,fila_bronze,order_id,fila_canonica,decision,regla_id,justificacion
0,7541,ORD-2026-00077,77,COPIA_EXCLUIDA,DUPLICADO_IDENTICO,Las 14 columnas originales coinciden con la pr...
1,7548,ORD-2026-00139,139,COPIA_EXCLUIDA,DUPLICADO_IDENTICO,Las 14 columnas originales coinciden con la pr...
2,7531,ORD-2026-00316,316,COPIA_EXCLUIDA,DUPLICADO_IDENTICO,Las 14 columnas originales coinciden con la pr...
3,7533,ORD-2026-00363,363,COPIA_EXCLUIDA,DUPLICADO_IDENTICO,Las 14 columnas originales coinciden con la pr...
4,7506,ORD-2026-00505,505,COPIA_EXCLUIDA,DUPLICADO_IDENTICO,Las 14 columnas originales coinciden con la pr...


## 4 · Evaluación problema por problema

Cada hallazgo recibe un estado. Los problemas sin regla aprobada permanecen `PENDIENTE`; las copias exactas reciben `EXCLUIDO_COMO_COPIA`.


In [5]:
def evaluar_acciones(problemas, decisiones):
    acciones = problemas.copy()
    mapa = decisiones.set_index("fila_bronze")["decision"] if not decisiones.empty else pd.Series(dtype="string")
    acciones["regla_id"] = ""
    acciones["estado_accion"] = "PENDIENTE"
    acciones["valor_preparado"] = acciones["valor_original"]
    acciones["justificacion"] = "No existe una regla aprobada para modificar este problema"
    es_dup = acciones["codigo_error"].eq("DUPLICADO")
    decision = acciones["fila_bronze"].astype(int).map(mapa).fillna("PENDIENTE")
    excluida = es_dup & decision.eq("COPIA_EXCLUIDA")
    acciones.loc[excluida, "regla_id"] = "DUPLICADO_IDENTICO"
    acciones.loc[excluida, "estado_accion"] = "EXCLUIDO_COMO_COPIA"
    acciones.loc[excluida, "justificacion"] = "Copia exacta retenida para auditoría y excluida del conjunto utilizable"
    acciones["version_tratamiento"] = VERSION_TRATAMIENTO
    return acciones

acciones = evaluar_acciones(problemas, decisiones_duplicados)
pendientes = acciones.loc[acciones["estado_accion"].eq("PENDIENTE")].groupby("fila_bronze")["columna_afectada"].agg(lambda x: "|".join(dict.fromkeys(x)))
df_trabajo["columnas_pendientes"] = df_trabajo["fila_bronze"].map(pendientes).fillna("")
df_trabajo["en_cuarentena_final"] = df_trabajo["columnas_pendientes"].ne("") | df_trabajo["excluido_como_copia"]
df_trabajo["registro_utilizable"] = ~df_trabajo["en_cuarentena_final"] & df_trabajo["registro_canonico"]
df_trabajo["version_tratamiento"] = VERSION_TRATAMIENTO

cuarentena_final = df_trabajo.loc[df_trabajo["en_cuarentena_final"]].copy()
# Silver de consumo: solo registros utilizables y columnas de negocio preparadas.
df_silver = df_trabajo.loc[df_trabajo["registro_utilizable"], [f"{col}_preparado" for col in COLUMNAS_ORIGINALES]].copy()
df_silver.columns = COLUMNAS_ORIGINALES

print("Utilizables:", int(df_trabajo["registro_utilizable"].sum()))
print("Cuarentena final:", len(cuarentena_final))


Utilizables: 7279
Cuarentena final: 271


## 5 · Validaciones y reporte

Las comprobaciones impiden perder filas, alterar los campos Bronze o admitir pedidos duplicados en el conjunto utilizable.


In [6]:
def construir_reporte():
    datos = [
        ("archivo_bronze", RUTAS["bronze"].name),
        ("sha256_bronze", hash_bronze),
        ("version_tratamiento", VERSION_TRATAMIENTO),
        ("filas_totales", len(df_trabajo)),
        ("filas_utilizables", int(df_trabajo["registro_utilizable"].sum())),
        ("filas_cuarentena_final", int(df_trabajo["en_cuarentena_final"].sum())),
        ("copias_exactas_excluidas", int(df_trabajo["excluido_como_copia"].sum())),
        ("problemas_pendientes", int(acciones["estado_accion"].eq("PENDIENTE").sum())),
    ]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

reporte = construir_reporte()
pd.testing.assert_frame_equal(df_trabajo[COLUMNAS_ORIGINALES], bronze[COLUMNAS_ORIGINALES])
assert len(df_trabajo) == len(bronze)
assert df_trabajo["fila_bronze"].is_unique
assert set(filas_excluidas).issubset(set(df_trabajo.loc[df_trabajo["en_cuarentena_final"], "fila_bronze"].astype(int)))
assert not df_trabajo.loc[df_trabajo["registro_utilizable"]].duplicated("order_id").any()
assert len(cuarentena_final) == int(df_trabajo["en_cuarentena_final"].sum())
assert len(df_silver) == int(df_trabajo["registro_utilizable"].sum())
assert list(df_silver.columns) == COLUMNAS_ORIGINALES
assert not df_silver.duplicated("order_id").any()

display(reporte)
print("Validaciones correctas")


,metrica,valor
0,archivo_bronze,andinalog_wms_orders.csv
1,sha256_bronze,d3b0132a66c8d65875a5d06691f8d9517b9a60db2f8d2a...
2,version_tratamiento,GIAD-M3-S4-WMS-orders-tratamiento-v1
3,filas_totales,7550
4,filas_utilizables,7279
5,filas_cuarentena_final,271
6,copias_exactas_excluidas,49
7,problemas_pendientes,225


Validaciones correctas


## 6 · Exportación reproducible

Las salidas se escriben primero en archivos temporales y reemplazan versiones anteriores. El Bronze, el diagnóstico y el catálogo nunca se sobrescriben.


In [7]:
def exportar_csvs(directorio, tablas):
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_wms2_", dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        if hashlib.sha256(RUTAS["bronze"].read_bytes()).hexdigest() != hash_bronze:
            raise RuntimeError("El Bronze cambió durante la ejecución")
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

salidas = exportar_csvs(RUTAS["salidas"], {
    "andinalog_wms_orders_tratado.csv": df_trabajo,
    "andinalog_wms_orders_silver.csv": df_silver,
    "andinalog_wms_orders_acciones.csv": acciones,
    "andinalog_wms_orders_decisiones_duplicados.csv": decisiones_duplicados,
    "andinalog_wms_orders_cuarentena_final.csv": cuarentena_final,
    "andinalog_wms_orders_reporte_tratamiento.csv": reporte,
})
for ruta in salidas:
    print(ruta)


c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_wms_orders\notebook2\salidas\andinalog_wms_orders_tratado.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_wms_orders\notebook2\salidas\andinalog_wms_orders_silver.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_wms_orders\notebook2\salidas\andinalog_wms_orders_acciones.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_wms_orders\notebook2\salidas\andinalog_wms_orders_decisiones_duplicados.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_wms_orders\notebook2\salidas\andinalog_wms_orders_cuarentena_final.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_wms_orders\notebook2\salidas\andinalog_wms_orders_reporte_tratamiento.csv


## Revisión posterior

Las reglas pendientes deben discutirse y aprobarse antes de cambiar su estado en el catálogo. El notebook no interpreta `cincuenta`, no corrige fechas ni modifica tiempos negativos mientras esas decisiones sigan pendientes.
